# 03 · Capa Gold e indicadores

**Qué hace:** a partir de Silver calcula tablas **ya resumidas** (una por pregunta de negocio). Estas son las que consume el dashboard.
Las tablas Gold son pequeñas, por eso el dashboard responde al instante.

In [ ]:
%run ./00_utilidades

In [ ]:
silver = leer_tabla(T_SILVER_1X)

# Código de departamento = 2 primeros dígitos del ubigeo (el ubigeo de 5 dígitos perdió el cero inicial)
silver = silver.withColumn("cod_departamento",
    F.when(F.col("ubigeo").rlike("^[0-9]{5,6}$"), F.substring(F.lpad(F.col("ubigeo"), 6, "0"), 1, 2)).otherwise("IGNORADO"))

## Tabla Gold 1: indicadores por año

In [ ]:
gold_anual = calcular_kpis(silver, "anio").orderBy("anio")
guardar_tabla(gold_anual, "gold_kpi_anual")
display(leer_tabla("gold_kpi_anual"))

## Tabla Gold 2: indicadores por departamento
Si subes `Ubigeo-2016.csv` a la carpeta `dataSet` del volumen, se agrega el nombre del departamento. Si no, queda el código.

In [ ]:
gold_dep = calcular_kpis(silver, "cod_departamento")
try:
    cat = (spark.read.csv(f"{RUTA_VOL}/dataSet/Ubigeo-2016.csv", header=True, inferSchema=False, sep=",")
           .select(F.substring(F.lpad(F.trim(F.col("Ubigeo")), 6, "0"), 1, 2).alias("cod_departamento"),
                   F.upper(F.trim(F.col("Departamento"))).alias("departamento"))
           .dropDuplicates(["cod_departamento"]))
    gold_dep = gold_dep.join(cat, "cod_departamento", "left").na.fill({"departamento": "IGNORADO"})
except Exception as ex:
    print("Sin catálogo de ubigeo; se usa solo el código de departamento.", str(ex)[:120])
guardar_tabla(gold_dep, "gold_kpi_departamento")
display(leer_tabla("gold_kpi_departamento").orderBy(F.desc("i1_total_nacimientos")))

## Tabla Gold 3: madres adolescentes por nivel de instrucción

In [ ]:
gold_inst = calcular_kpis(silver, "nivel_instruccion", "grupo_edad")
guardar_tabla(gold_inst, "gold_kpi_instruccion_edad")
display(leer_tabla("gold_kpi_instruccion_edad").orderBy("nivel_instruccion", "grupo_edad"))

## Tabla Gold 4: bajo peso según semanas de gestación

In [ ]:
gold_sem = calcular_kpis(silver.where("semanas_gestacion IS NOT NULL"), "semanas_gestacion").orderBy("semanas_gestacion")
guardar_tabla(gold_sem, "gold_bajo_peso_semanas")
display(leer_tabla("gold_bajo_peso_semanas"))

## Tabla Gold 5: tipo de parto

In [ ]:
gold_parto = calcular_kpis(silver, "tipo_parto")
guardar_tabla(gold_parto, "gold_kpi_tipo_parto")
display(leer_tabla("gold_kpi_tipo_parto"))

## Dashboard en Databricks (paso a paso)
1. En el menú izquierdo: **Dashboards → Create dashboard**.
2. En **Data** agrega estos *datasets* (uno por consulta):
   - `SELECT * FROM workspace.experimentobigdata.gold_kpi_anual ORDER BY anio`
   - `SELECT * FROM workspace.experimentobigdata.gold_kpi_departamento`
   - `SELECT * FROM workspace.experimentobigdata.gold_kpi_instruccion_edad`
   - `SELECT * FROM workspace.experimentobigdata.gold_bajo_peso_semanas`
3. En **Canvas** crea: 3 contadores (I1, I2, I3), línea de I1 por año, barras de I2 por departamento, barras de I3 por nivel de instrucción y columnas de I2 por semanas.
4. Publica el dashboard y toma capturas para el informe (evidencia).

**Genie (opcional):** crea un *Genie space* con las tablas `gold_*` y pregunta, por ejemplo: «¿en qué año hubo más nacimientos?» o «¿qué departamento tiene el mayor % de bajo peso?».
Si lo usas, agrégalo a la declaración de uso de IA del informe.